In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # JobFlow AI — Matching entre Perfil e Vagas
# MAGIC
# MAGIC Este notebook cria um perfil sintético de usuário e calcula
# MAGIC um score inicial de compatibilidade entre o perfil e as vagas.
# MAGIC
# MAGIC Saídas:
# MAGIC
# MAGIC - demo_user_profiles
# MAGIC - demo_user_profile_skills
# MAGIC - gold_job_match_scores

# COMMAND ----------

from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql import types as T

# COMMAND ----------

CATALOG = "workspace"
SCHEMA = "jobflow_ai"

GOLD_JOBS_TABLE = f"{CATALOG}.{SCHEMA}.gold_job_postings"
JOB_SKILLS_TABLE = f"{CATALOG}.{SCHEMA}.gold_job_skill_matches"

USER_PROFILES_TABLE = f"{CATALOG}.{SCHEMA}.demo_user_profiles"
USER_PROFILE_SKILLS_TABLE = f"{CATALOG}.{SCHEMA}.demo_user_profile_skills"
MATCH_SCORES_TABLE = f"{CATALOG}.{SCHEMA}.gold_job_match_scores"

DEMO_USER_ID = "demo_user_001"
DEMO_PROFILE_ID = "demo_profile_data_engineer"

print("=" * 70)
print("JOBFLOW AI — MATCHING PERFIL X VAGAS")
print("=" * 70)
print(f"Gold jobs table: {GOLD_JOBS_TABLE}")
print(f"Job skills table: {JOB_SKILLS_TABLE}")
print(f"User profiles table: {USER_PROFILES_TABLE}")
print(f"User profile skills table: {USER_PROFILE_SKILLS_TABLE}")
print(f"Match scores table: {MATCH_SCORES_TABLE}")
print(f"Horário UTC: {datetime.now(timezone.utc).isoformat()}")
print("=" * 70)

# COMMAND ----------

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

jobs_df = spark.table(GOLD_JOBS_TABLE)
job_skills_df = spark.table(JOB_SKILLS_TABLE)

print(f"Vagas Gold: {jobs_df.count()}")
print(f"Skills detectadas em vagas: {job_skills_df.count()}")

display(jobs_df.limit(5))
display(job_skills_df.limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Criar perfil sintético do usuário

# COMMAND ----------

profile_rows = [
    {
        "user_id": DEMO_USER_ID,
        "profile_id": DEMO_PROFILE_ID,
        "full_name": "Demo Data Engineer",
        "headline": "Data Engineer buscando vagas remotas com Python, SQL, Spark e Databricks",
        "target_roles": ["data engineer", "analytics engineer", "backend engineer"],
        "preferred_location": "remote",
        "remote_preference": "remote",
        "min_salary": 70000,
        "salary_currency": "USD",
        "seniority_target": "mid",
        "created_at": datetime.now(timezone.utc).replace(tzinfo=None),
    }
]

profile_schema = T.StructType(
    [
        T.StructField("user_id", T.StringType(), False),
        T.StructField("profile_id", T.StringType(), False),
        T.StructField("full_name", T.StringType(), False),
        T.StructField("headline", T.StringType(), True),
        T.StructField("target_roles", T.ArrayType(T.StringType()), True),
        T.StructField("preferred_location", T.StringType(), True),
        T.StructField("remote_preference", T.StringType(), True),
        T.StructField("min_salary", T.LongType(), True),
        T.StructField("salary_currency", T.StringType(), True),
        T.StructField("seniority_target", T.StringType(), True),
        T.StructField("created_at", T.TimestampType(), False),
    ]
)

profiles_df = spark.createDataFrame(profile_rows, schema=profile_schema)

(
    profiles_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(USER_PROFILES_TABLE)
)

display(profiles_df)

print(f"OK: perfil sintético gravado em {USER_PROFILES_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Criar skills do perfil

# COMMAND ----------

profile_skill_rows = [
    ("python", "advanced", 5),
    ("sql", "advanced", 5),
    ("spark", "intermediate", 3),
    ("databricks", "intermediate", 2),
    ("aws", "intermediate", 3),
    ("docker", "basic", 1),
    ("git", "intermediate", 4),
    ("etl", "advanced", 5),
    ("analytics", "intermediate", 3),
    ("api", "intermediate", 3),
]

profile_skills_schema = T.StructType(
    [
        T.StructField("skill_name", T.StringType(), False),
        T.StructField("skill_level", T.StringType(), False),
        T.StructField("years_experience", T.IntegerType(), True),
    ]
)

profile_skills_base_df = spark.createDataFrame(
    profile_skill_rows,
    schema=profile_skills_schema,
)

profile_skills_df = (
    profile_skills_base_df
    .withColumn("user_id", F.lit(DEMO_USER_ID))
    .withColumn("profile_id", F.lit(DEMO_PROFILE_ID))
    .withColumn("created_at", F.current_timestamp())
    .select(
        "user_id",
        "profile_id",
        "skill_name",
        "skill_level",
        "years_experience",
        "created_at",
    )
)

(
    profile_skills_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(USER_PROFILE_SKILLS_TABLE)
)

display(profile_skills_df)

print(f"OK: skills do perfil gravadas em {USER_PROFILE_SKILLS_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Agregar skills por vaga

# COMMAND ----------

job_skill_summary_df = (
    job_skills_df
    .groupBy("job_id")
    .agg(
        F.collect_set("skill_name").alias("job_skills"),
        F.countDistinct("skill_name").alias("job_skill_count"),
        F.collect_set(
            F.when(F.col("requirement_type") == "required", F.col("skill_name"))
        ).alias("required_skills_raw"),
        F.collect_set(
            F.when(F.col("requirement_type") == "preferred", F.col("skill_name"))
        ).alias("preferred_skills_raw"),
    )
    .withColumn(
        "required_skills",
        F.expr("filter(required_skills_raw, x -> x is not null)")
    )
    .withColumn(
        "preferred_skills",
        F.expr("filter(preferred_skills_raw, x -> x is not null)")
    )
    .drop("required_skills_raw", "preferred_skills_raw")
)

display(job_skill_summary_df.limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Preparar perfil agregado

# COMMAND ----------

profile_agg_df = (
    profile_skills_df
    .groupBy("user_id", "profile_id")
    .agg(
        F.collect_set("skill_name").alias("profile_skills"),
        F.countDistinct("skill_name").alias("profile_skill_count"),
    )
)

display(profile_agg_df)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Calcular score inicial de compatibilidade

# COMMAND ----------

match_df = (
    jobs_df
    .join(job_skill_summary_df, on="job_id", how="left")
    .crossJoin(profile_agg_df)
    .withColumn(
        "job_skills",
        F.coalesce(F.col("job_skills"), F.array().cast("array<string>"))
    )
    .withColumn(
        "required_skills",
        F.coalesce(F.col("required_skills"), F.array().cast("array<string>"))
    )
    .withColumn(
        "preferred_skills",
        F.coalesce(F.col("preferred_skills"), F.array().cast("array<string>"))
    )
    .withColumn(
        "matched_skills",
        F.array_intersect(F.col("job_skills"), F.col("profile_skills"))
    )
    .withColumn(
        "missing_skills",
        F.array_except(F.col("job_skills"), F.col("profile_skills"))
    )
    .withColumn(
        "matched_required_skills",
        F.array_intersect(F.col("required_skills"), F.col("profile_skills"))
    )
    .withColumn(
        "missing_required_skills",
        F.array_except(F.col("required_skills"), F.col("profile_skills"))
    )
    .withColumn(
        "matched_skill_count",
        F.size(F.col("matched_skills"))
    )
    .withColumn(
        "missing_skill_count",
        F.size(F.col("missing_skills"))
    )
    .withColumn(
        "required_skill_count",
        F.size(F.col("required_skills"))
    )
    .withColumn(
        "missing_required_skill_count",
        F.size(F.col("missing_required_skills"))
    )
    .withColumn(
        "skill_coverage_score",
        F.when(
            F.col("job_skill_count") > 0,
            F.col("matched_skill_count") / F.col("job_skill_count")
        ).otherwise(F.lit(0.2))
    )
    .withColumn(
        "required_skill_score",
        F.when(
            F.col("required_skill_count") > 0,
            F.size(F.col("matched_required_skills")) / F.col("required_skill_count")
        ).otherwise(F.lit(0.7))
    )
    .withColumn(
        "remote_score",
        F.when(F.col("remote_type") == "remote", F.lit(1.0))
        .otherwise(F.lit(0.5))
    )
    .withColumn(
        "salary_score",
        F.when(F.col("salary_min").isNull() & F.col("salary_max").isNull(), F.lit(0.6))
        .when(F.col("salary_max") >= F.lit(70000), F.lit(1.0))
        .when(F.col("salary_min") >= F.lit(70000), F.lit(1.0))
        .otherwise(F.lit(0.4))
    )
    .withColumn(
        "data_quality_score_safe",
        F.coalesce(F.col("data_quality_score"), F.lit(0.5))
    )
    .withColumn(
        "match_score",
        F.round(
            (
                F.col("skill_coverage_score") * F.lit(0.45)
                + F.col("required_skill_score") * F.lit(0.25)
                + F.col("remote_score") * F.lit(0.15)
                + F.col("salary_score") * F.lit(0.10)
                + F.col("data_quality_score_safe") * F.lit(0.05)
            ) * F.lit(100),
            2,
        )
    )
    .withColumn(
        "match_tier",
        F.when(F.col("match_score") >= 80, F.lit("strong"))
        .when(F.col("match_score") >= 60, F.lit("good"))
        .when(F.col("match_score") >= 40, F.lit("partial"))
        .otherwise(F.lit("low"))
    )
    .withColumn(
        "match_explanation",
        F.concat_ws(
            " ",
            F.concat(F.lit("Score: "), F.col("match_score").cast("string"), F.lit("/100.")),
            F.concat(F.lit("Skills encontradas: "), F.concat_ws(", ", F.col("matched_skills")), F.lit(".")),
            F.concat(F.lit("Skills ausentes: "), F.concat_ws(", ", F.col("missing_skills")), F.lit(".")),
            F.concat(F.lit("Tipo remoto: "), F.col("remote_type"), F.lit(".")),
        )
    )
    .withColumn(
        "match_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("user_id"),
                F.col("profile_id"),
                F.col("job_id"),
            ),
            256,
        )
    )
    .withColumn("matched_at", F.current_timestamp())
    .select(
        "match_id",
        "user_id",
        "profile_id",
        "job_id",
        "source_system",
        "source_job_id",
        "job_title",
        "company_name",
        "job_location",
        "remote_type",
        "salary_min",
        "salary_max",
        "salary_currency",
        "job_url",
        "apply_url",
        "profile_skills",
        "job_skills",
        "required_skills",
        "preferred_skills",
        "matched_skills",
        "missing_skills",
        "matched_required_skills",
        "missing_required_skills",
        "job_skill_count",
        "matched_skill_count",
        "missing_skill_count",
        "required_skill_count",
        "missing_required_skill_count",
        "skill_coverage_score",
        "required_skill_score",
        "remote_score",
        "salary_score",
        "data_quality_score_safe",
        "match_score",
        "match_tier",
        "match_explanation",
        "matched_at",
    )
)

display(
    match_df.select(
        "job_title",
        "company_name",
        "remote_type",
        "matched_skills",
        "missing_skills",
        "match_score",
        "match_tier",
    )
    .orderBy(F.col("match_score").desc())
    .limit(20)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Gravar tabela de scores

# COMMAND ----------

(
    match_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(MATCH_SCORES_TABLE)
)

print(f"OK: scores gravados em {MATCH_SCORES_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Validações

# COMMAND ----------

scores_df = spark.table(MATCH_SCORES_TABLE)

summary_df = scores_df.agg(
    F.count("*").alias("records"),
    F.countDistinct("job_id").alias("distinct_jobs"),
    F.round(F.avg("match_score"), 2).alias("avg_match_score"),
    F.max("match_score").alias("max_match_score"),
    F.min("match_score").alias("min_match_score"),
    F.sum(F.when(F.col("match_tier") == "strong", 1).otherwise(0)).alias("strong_matches"),
    F.sum(F.when(F.col("match_tier") == "good", 1).otherwise(0)).alias("good_matches"),
    F.sum(F.when(F.col("match_tier") == "partial", 1).otherwise(0)).alias("partial_matches"),
    F.sum(F.when(F.col("match_tier") == "low", 1).otherwise(0)).alias("low_matches"),
)

display(summary_df)

display(
    scores_df.groupBy("match_tier")
    .agg(F.count("*").alias("jobs"))
    .orderBy(
        F.when(F.col("match_tier") == "strong", 1)
        .when(F.col("match_tier") == "good", 2)
        .when(F.col("match_tier") == "partial", 3)
        .otherwise(4)
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Top vagas recomendadas

# COMMAND ----------

display(
    scores_df.select(
        "match_score",
        "match_tier",
        "job_title",
        "company_name",
        "job_location",
        "remote_type",
        "salary_min",
        "salary_max",
        "matched_skills",
        "missing_skills",
        "match_explanation",
        "job_url",
    )
    .orderBy(F.col("match_score").desc(), F.col("job_title"))
    .limit(20)
)

# COMMAND ----------

print()
print("=" * 70)
print("RESULTADO: MATCHING PERFIL X VAGAS CONCLUÍDO")
print("=" * 70)
print(f"user profiles table: {USER_PROFILES_TABLE}")
print(f"profile skills table: {USER_PROFILE_SKILLS_TABLE}")
print(f"match scores table: {MATCH_SCORES_TABLE}")
print(f"records: {scores_df.count()}")
print(f"top score: {scores_df.agg(F.max('match_score')).first()[0]}")
print("=" * 70)